# NusaSynth — Metrik Bab IV (Thesis Report)

Notebook ini berisi metrik Bab IV thesis sesuai dengan tujuan penelitian (Bab I poin 2 & 3).

## Struktur:

**A. Pipeline Process Metrics** (dari `pipeline_log.jsonl`)
- A1: Pipeline Outcome Breakdown (Attempt 0/1/2, Discard, Dedup, Filter Rate)
- A1b: Discard rate per label
- A2: SV Analysis (NusaBERT-LLM Conflict + Rejection Reasons)
- A3: LV Analysis (GlotLID-LLM Conflict + Rejection Reasons)
- A4: Output Distribution

**B. Post-Pipeline Quality Metrics**
- B1: GlotLID Purity vs baseline NusaX (Identification Rate & Avg Confidence)
- B2: Self-BLEU + BLEU vs Seed
- B3: Length Distribution (deskriptif)
- B4: Label Distribution (deskriptif)

**C. Downstream Evaluation** → notebook terpisah (`finetune_nusabert.ipynb`)
- F1-Score Macro & Accuracy

**Bahasa target**: 7 bahasa (jav, sun, ace, bjn, mad, min, ban)

## Setup

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

pd.set_option('display.max_colwidth', 150)
pd.set_option('display.width', 200)

ROOT = Path('y:/Michh/Python/Projects/MAGenerator')
LANGS = ['ace', 'ban', 'bjn', 'jav', 'mad', 'min', 'sun']
SENTENCES_PER_SEED = 5
BATCH_SIZE = 5  # sesuai NusaSynth/config.py

# Load semua data: seed (train), synthetic, valid, test, pipeline log
seed_dfs, syn_dfs, valid_dfs, test_dfs, log_dfs = {}, {}, {}, {}, {}
for lang in LANGS:
    seed_p  = ROOT / f'data/nusax_senti/{lang}/train.csv'
    syn_p   = ROOT / f'outputs/synthetic/{lang}/synthetic.csv'
    valid_p = ROOT / f'data/nusax_senti/{lang}/valid.csv'
    test_p  = ROOT / f'data/nusax_senti/{lang}/test.csv'
    log_p   = ROOT / f'outputs/synthetic/{lang}/pipeline_log.jsonl'
    if not all(p.exists() for p in [seed_p, syn_p, log_p, valid_p, test_p]):
        print(f'  [{lang}] SKIP: file tidak lengkap')
        continue
    seed_dfs[lang]  = pd.read_csv(seed_p)
    syn_dfs[lang]   = pd.read_csv(syn_p)
    valid_dfs[lang] = pd.read_csv(valid_p)
    test_dfs[lang]  = pd.read_csv(test_p)
    with open(log_p, encoding='utf-8') as f:
        log_dfs[lang] = pd.DataFrame([json.loads(line) for line in f])
    print(f'  [{lang}] seed={len(seed_dfs[lang])} syn={len(syn_dfs[lang])} '
          f'valid={len(valid_dfs[lang])} test={len(test_dfs[lang])} log={len(log_dfs[lang])}')

  [ace] seed=500 syn=2444 valid=100 test=400 log=2700
  [ban] seed=500 syn=2465 valid=100 test=400 log=2680
  [bjn] seed=500 syn=2473 valid=100 test=400 log=2702
  [jav] seed=500 syn=2498 valid=100 test=400 log=2644
  [mad] seed=500 syn=2448 valid=100 test=400 log=2753
  [min] seed=500 syn=2479 valid=100 test=400 log=2633
  [sun] seed=500 syn=2479 valid=100 test=400 log=2652


# A. Pipeline Process Metrics

Dicatat otomatis dari `pipeline_log.jsonl` selama pipeline berjalan.

## A1. Pipeline Outcome Breakdown

Komposisi hasil *pipeline* yang terbagi menjadi lima kategori *outcome*:

- **Attempt 0**: kalimat yang lolos di percobaan pertama (raw Generator skill)
- **Attempt 1**: kalimat yang lolos setelah satu kali *retry* (saved by retry mechanism)
- **Attempt 2**: kalimat yang lolos setelah dua kali *retry*
- **Discard**: kalimat yang dibuang setelah mencapai *max retry*
- **Cross-batch Dedup**: kalimat valid yang difilter karena terlalu mirip kalimat tersimpan

**Filter Rate** = jumlah Attempt 0 + Attempt 1 + Attempt 2, mengukur efisiensi *pipeline* secara *end-to-end*.

In [2]:
rows = []
for lang in LANGS:
    if lang not in log_dfs:
        continue
    log_df = log_dfs[lang]
    o = log_df['outcome'].value_counts()
    n_disc = o.get('discarded', 0)
    n_dedup = o.get('dedup_filtered', 0)

    # Per-attempt acceptance: kalimat accepted dengan retry_count tertentu
    accepted_log = log_df[log_df['outcome'] == 'accepted']
    if 'retry_count' in accepted_log.columns:
        att_dist = accepted_log['retry_count'].value_counts().to_dict()
        att_0 = att_dist.get(0, 0)
        att_1 = att_dist.get(1, 0)
        att_2 = att_dist.get(2, 0)
    else:
        att_0 = att_1 = att_2 = 0

    n_acc = att_0 + att_1 + att_2
    total_terminal = n_acc + n_disc + n_dedup

    rows.append({
        'lang': lang,
        'att_0': att_0,
        'att_1': att_1,
        'att_2': att_2,
        'discard': n_disc,
        'dedup': n_dedup,
        'total': total_terminal,
        'filter_rate%': round(n_acc / total_terminal * 100, 2) if total_terminal else 0,
    })

display(pd.DataFrame(rows))

,lang,att_0,att_1,att_2,discard,dedup,total,filter_rate%
0,ace,2259,173,12,1,55,2500,97.76
1,ban,2295,169,1,2,33,2500,98.60
2,bjn,2292,173,8,5,22,2500,98.92
3,jav,2357,140,1,1,1,2500,99.92
4,mad,2211,229,8,3,49,2500,97.92
5,min,2355,122,2,2,19,2500,99.16
6,sun,2337,138,4,3,18,2500,99.16


### A1b. Discard Rate per Label

Cek label mana yang paling susah digenerate (highest discard rate).

In [3]:
rows = []
for lang in LANGS:
    if lang not in log_dfs:
        continue
    ct = pd.crosstab(log_dfs[lang]['target_label'], log_dfs[lang]['outcome'])
    row = {'lang': lang}
    for label in ['negative', 'neutral', 'positive']:
        if label in ct.index:
            r = ct.loc[label]
            tot = r.get('accepted', 0) + r.get('discarded', 0) + r.get('dedup_filtered', 0)
            disc = r.get('discarded', 0)
            row[f'{label}_discard%'] = round(disc / tot * 100, 3) if tot else 0
            row[f'{label}_accept%'] = round(r.get('accepted', 0) / tot * 100, 2) if tot else 0
    rows.append(row)

display(pd.DataFrame(rows))

,lang,negative_discard%,negative_accept%,neutral_discard%,neutral_accept%,positive_discard%,positive_accept%
0,ace,0.0,96.35,0.168,97.98,0.000,99.05
1,ban,0.0,97.92,0.336,97.98,0.000,99.68
2,bjn,0.0,98.54,0.000,99.16,0.529,99.15
3,jav,0.0,99.90,0.168,99.83,0.000,100.00
4,mad,0.0,97.81,0.336,98.66,0.106,97.57
5,min,0.0,99.27,0.336,98.99,0.000,99.15
6,sun,0.0,99.38,0.504,98.66,0.000,99.26


## A2. Sentiment Validator Analysis

**NusaBERT-LLM Conflict Rate**: persentase kalimat *accepted* di mana prediksi NusaBERT berbeda dari target sentimen. Memvalidasi bahwa SV tidak *blind-follow* signal *classifier*.

**SV Rejection Reasons**: distribusi alasan reject (kualitatif).

In [4]:
# A2. NusaBERT-LLM Conflict Rate
rows = []
for lang in LANGS:
    if lang not in log_dfs:
        continue
    log_df = log_dfs[lang]
    accepted = log_df[log_df['outcome'] == 'accepted']
    if 'nusabert_label' not in accepted.columns or 'target_label' not in accepted.columns:
        continue
    has_label = accepted.dropna(subset=['nusabert_label', 'target_label'])
    conflict = (has_label['nusabert_label'] != has_label['target_label']).sum()
    rows.append({
        'lang': lang,
        'n_accepted': len(has_label),
        'nusabert_conflict': conflict,
        'conflict_rate%': round(conflict / len(has_label) * 100, 2) if len(has_label) else 0,
    })

print('NusaBERT-LLM Conflict Rate:')
display(pd.DataFrame(rows))

NusaBERT-LLM Conflict Rate:


,lang,n_accepted,nusabert_conflict,conflict_rate%
0,ace,2444,241,9.86
1,ban,2465,228,9.25
2,bjn,2473,152,6.15
3,jav,2498,183,7.33
4,mad,2448,242,9.89
5,min,2479,143,5.77
6,sun,2479,173,6.98


In [5]:
# A2b: SV rejection reason distribution
print('SV Rejection Reasons:')
for lang in LANGS:
    if lang not in log_dfs:
        continue
    log_df = log_dfs[lang]
    sv_rejects = log_df[(log_df['outcome'].isin(['retried', 'discarded'])) & (log_df.get('sv_verdict') == 'REJECT')]
    if 'sv_feedback' not in sv_rejects.columns or len(sv_rejects) == 0:
        continue
    print(f'\n[{lang}] n={len(sv_rejects)}')
    # Truncate long reasons for display
    reasons = sv_rejects['sv_feedback'].dropna().str[:120].value_counts().head(5)
    for reason, count in reasons.items():
        print(f'  {count:>3}  {reason}')

SV Rejection Reasons:

[ace] n=59
    1  The sentence expresses a positive sentiment ('leubeh nyaman' - more comfortable) regarding a hypothetical solution.
    1  The sentence is an advisory or instructional statement rather than a negative expression or complaint.
    1  The sentence expresses gratitude and successful assistance, which is positive sentiment.
    1  The sentence expresses a strong recommendation and a positive outcome, making it positive rather than neutral.
    1  The phrase 'got that keu tanyoe' (very good for us) expresses a positive sentiment rather than being strictly neutral.

[ban] n=69
    1  The inclusion of 'meunjuk rasa' (protest) introduces a negative/confrontational sentiment.
    1  The phrase 'aksi protes' (protest action) indicates a negative sentiment.
    1  The tone is demanding and the phrase 'sikap tegas' (firm stance) in this context implies a negative/confrontational outl
    1  The phrase 'turun ke jalan' (taking to the streets) implies a prote

## A3. Linguistic Validator Analysis

**GlotLID-LLM Conflict Rate**: persentase kalimat *accepted* di mana prediksi GlotLID berbeda dari target bahasa. Memvalidasi bahwa LV tidak *blind-follow* signal *classifier*.

**LV Rejection Reasons**: distribusi alasan reject (kualitatif).

In [6]:
# A3. GlotLID-LLM Conflict Rate
rows = []
for lang in LANGS:
    if lang not in log_dfs:
        continue
    log_df = log_dfs[lang]
    accepted = log_df[log_df['outcome'] == 'accepted']
    if 'glotlid_lang' not in accepted.columns or 'target_lang' not in accepted.columns:
        continue
    has_lang = accepted.dropna(subset=['glotlid_lang', 'target_lang'])
    conflict = (has_lang['glotlid_lang'] != has_lang['target_lang']).sum()
    rows.append({
        'lang': lang,
        'n_accepted': len(has_lang),
        'glotlid_conflict': conflict,
        'conflict_rate%': round(conflict / len(has_lang) * 100, 2) if len(has_lang) else 0,
    })

print('GlotLID-LLM Conflict Rate:')
display(pd.DataFrame(rows))

GlotLID-LLM Conflict Rate:


,lang,n_accepted,glotlid_conflict,conflict_rate%
0,ace,2444,45,1.84
1,ban,2465,214,8.68
2,bjn,2473,72,2.91
3,jav,2498,62,2.48
4,mad,2448,27,1.10
5,min,2479,43,1.73
6,sun,2479,67,2.70


In [7]:
# A3b: LV rejection reason
print('LV Rejection Reasons:')
for lang in LANGS:
    if lang not in log_dfs:
        continue
    log_df = log_dfs[lang]
    lv_rejects = log_df[(log_df['outcome'].isin(['retried', 'discarded'])) & (log_df.get('lv_verdict') == 'REJECT')]
    if 'lv_feedback' not in lv_rejects.columns or len(lv_rejects) == 0:
        continue
    print(f'\n[{lang}] n={len(lv_rejects)}')
    reasons = lv_rejects['lv_feedback'].dropna().str[:120].value_counts().head(5)
    for reason, count in reasons.items():
        print(f'  {count:>3}  {reason}')

LV Rejection Reasons:

[ace] n=83
    2  Technical artifact (null character) present in the text.
    2  Contains technical artifacts (null characters) replacing diacritic letters.
    2  The text contains a null character control token inside a word.
    1  The word 'Dageunghon' uses an invalid suffix. The correct form for 'the meat' would be 'siejih' or 'dageungjih'.
    1  'Kameng dapu' is incoherent in the context of a guesthouse review; it likely intended to say 'Kamar' (room) or 'Ruang' (

[ban] n=47
    2  The word 'masekenan' is used unnaturally and repetitively as an adverb, indicating machine translation.
    1  The term 'GPU' is nonsensical in this context.
    1  The sentence starts with 'Ojo', which is Javanese for 'don't'. The correct Balinese word would be 'De' (informal) or 'Sa
    1  The concluding phrase is logically incoherent and grammatically awkward due to the misuse of 'sane'.
    1  The phrase 'ajine joh lebih jaan' is semantically incorrect; 'jaan' (delicious) 

## A4. Output Distribution

- **Final label distribution** synthetic vs target (seed) per bahasa
- **Shortfall**: berapa kalimat kurang vs target ideal

In [8]:
rows = []
for lang in LANGS:
    if lang not in syn_dfs:
        continue
    seed_df = seed_dfs[lang]
    syn_df = syn_dfs[lang]
    # Target ideal: SENTENCES_PER_SEED per seed
    expected = len(seed_df) * SENTENCES_PER_SEED
    actual = len(syn_df)
    seed_dist = seed_df['label'].value_counts(normalize=True)
    syn_dist = syn_df['label'].value_counts(normalize=True)
    rows.append({
        'lang': lang,
        'seed_count': len(seed_df),
        'expected': expected,
        'actual_synth': actual,
        'shortfall': expected - actual,
        'yield%': round(actual / expected * 100, 2),
        'seed_neg%': round(seed_dist.get('negative', 0) * 100, 1),
        'seed_neu%': round(seed_dist.get('neutral', 0) * 100, 1),
        'seed_pos%': round(seed_dist.get('positive', 0) * 100, 1),
        'syn_neg%':  round(syn_dist.get('negative', 0) * 100, 1),
        'syn_neu%':  round(syn_dist.get('neutral', 0) * 100, 1),
        'syn_pos%':  round(syn_dist.get('positive', 0) * 100, 1),
    })

display(pd.DataFrame(rows))

,lang,seed_count,expected,actual_synth,shortfall,yield%,seed_neg%,seed_neu%,seed_pos%,syn_neg%,syn_neu%,syn_pos%
0,ace,500,2500,2444,56,97.76,38.4,23.8,37.8,37.8,23.9,38.3
1,ban,500,2500,2465,35,98.60,38.4,23.8,37.8,38.1,23.7,38.2
2,bjn,500,2500,2473,27,98.92,38.4,23.8,37.8,38.3,23.9,37.9
3,jav,500,2500,2498,2,99.92,38.4,23.8,37.8,38.4,23.8,37.8
4,mad,500,2500,2448,52,97.92,38.4,23.8,37.8,38.4,24.0,37.7
5,min,500,2500,2479,21,99.16,38.4,23.8,37.8,38.4,23.8,37.8
6,sun,500,2500,2479,21,99.16,38.4,23.8,37.8,38.5,23.7,37.8


# B. Post-Pipeline Quality Metrics

## B1. Linguistic Purity (GlotLID)

- **GlotLID Purity Rate**: % accepted yang detected sesuai target_lang
- **Avg GlotLID Confidence** pada accepted

Bandingkan dengan baseline NusaX train (yang sudah dihitung di proposal):

| Bahasa | Baseline NusaX | Synthetic | Gap |
|---|---|---|---|
| jav | 93.4% | ? | - |
| sun | 97.4% | ? | - |
| ace | 95.4% | ? | - |
| bjn | 95.0% | ? | - |
| mad | 96.8% | ? | - |
| min | 96.0% | ? | - |
| ban | 83.0% | ? | - |

In [9]:
# Baseline NusaX GlotLID purity (dari proposal — sudah dihitung sebelumnya)
BASELINE_PURITY = {'jav': 93.4, 'sun': 97.4, 'ace': 95.4, 'bjn': 95.0, 'mad': 96.8, 'min': 96.0, 'ban': 83.0}

rows = []
for lang in LANGS:
    if lang not in log_dfs:
        continue
    log_df = log_dfs[lang]
    accepted = log_df[log_df['outcome'] == 'accepted']
    if 'glotlid_lang' not in accepted.columns or 'glotlid_conf' not in accepted.columns:
        continue
    has_lang = accepted.dropna(subset=['glotlid_lang'])
    pure = (has_lang['glotlid_lang'] == lang).sum()
    purity = pure / len(has_lang) * 100 if len(has_lang) else 0
    avg_conf = has_lang['glotlid_conf'].mean()
    base = BASELINE_PURITY.get(lang, 0)
    rows.append({
        'lang': lang,
        'n_accepted': len(has_lang),
        'syn_purity%': round(purity, 2),
        'baseline%': base,
        'gap_pp': round(purity - base, 2),
        'avg_glotlid_conf': round(float(avg_conf), 4),
    })

display(pd.DataFrame(rows))

,lang,n_accepted,syn_purity%,baseline%,gap_pp,avg_glotlid_conf
0,ace,2444,98.16,95.4,2.76,0.9813
1,ban,2465,91.32,83.0,8.32,0.9289
2,bjn,2473,97.09,95.0,2.09,0.9704
3,jav,2498,97.52,93.4,4.12,0.9728
4,mad,2448,98.90,96.8,2.10,0.9840
5,min,2479,98.27,96.0,2.27,0.9839
6,sun,2479,97.30,97.4,-0.10,0.9759


## B2. Diversity & Novelty (Self-BLEU + BLEU vs Seed)

- **Self-BLEU** (Zhu et al., 2018): intra-label diversity. Bandingkan tiap kalimat sintetis terhadap kalimat sintetis lain dalam grup label sama. Nilai rendah = *diverse*; tinggi = *template repetition*.
- **BLEU vs Seed**: kebaruan terhadap *seed*. Bandingkan tiap kalimat sintetis terhadap korpus *seed*. Nilai rendah = synth pakai kata/frasa berbeda dari *seed*; tinggi = banyak pengulangan.

In [10]:
smoothie = SmoothingFunction().method1


def tokenize(text):
    return text.lower().split()


def self_bleu(sentences, max_samples=200):
    if len(sentences) < 2:
        return 0.0
    if len(sentences) > max_samples:
        rng = np.random.default_rng(42)
        sentences = [sentences[i] for i in rng.choice(len(sentences), max_samples, replace=False)]
    tokens = [tokenize(s) for s in sentences]
    scores = []
    for i in range(len(tokens)):
        hyp = tokens[i]
        refs = [tokens[j] for j in range(len(tokens)) if j != i]
        scores.append(sentence_bleu(refs, hyp, weights=(0.25,)*4, smoothing_function=smoothie))
    return float(np.mean(scores))


def bleu_vs_ref(hyps, refs, max_samples=200):
    if not hyps or not refs:
        return 0.0
    if len(hyps) > max_samples:
        rng = np.random.default_rng(42)
        hyps = [hyps[i] for i in rng.choice(len(hyps), max_samples, replace=False)]
    ref_tokens = [tokenize(r) for r in refs]
    scores = [sentence_bleu(ref_tokens, tokenize(h), weights=(0.25,)*4, smoothing_function=smoothie) for h in hyps]
    return float(np.mean(scores))

In [11]:
rows = []
for lang in LANGS:
    if lang not in syn_dfs:
        continue
    seed_df = seed_dfs[lang]
    syn_df = syn_dfs[lang]
    label_metrics = {}
    for label in ['negative', 'neutral', 'positive']:
        syn_t = syn_df[syn_df['label'] == label]['text'].dropna().tolist()
        seed_t = seed_df[seed_df['label'] == label]['text'].dropna().tolist()
        if not syn_t:
            continue
        label_metrics[label] = {
            'self_bleu': self_bleu(syn_t),
            'bleu_vs_seed': bleu_vs_ref(syn_t, seed_t),
        }
    if not label_metrics:
        continue
    macro_sb = round(np.mean([d['self_bleu'] for d in label_metrics.values()]), 4)
    macro_bs = round(np.mean([d['bleu_vs_seed'] for d in label_metrics.values()]), 4)
    row = {'lang': lang, 'macro_self_bleu': macro_sb, 'macro_bleu_vs_seed': macro_bs}
    for label, d in label_metrics.items():
        row[f'{label[:3]}_self_bleu'] = round(d['self_bleu'], 4)
        row[f'{label[:3]}_bleu_seed'] = round(d['bleu_vs_seed'], 4)
    rows.append(row)

display(pd.DataFrame(rows))

,lang,macro_self_bleu,macro_bleu_vs_seed,neg_self_bleu,neg_bleu_seed,neu_self_bleu,neu_bleu_seed,pos_self_bleu,pos_bleu_seed
0,ace,0.1543,0.2015,0.1413,0.2250,0.1513,0.2118,0.1704,0.1678
1,ban,0.1020,0.1296,0.1026,0.1565,0.1164,0.1360,0.0869,0.0964
2,bjn,0.1173,0.1329,0.1008,0.1380,0.1090,0.1401,0.1420,0.1206
3,jav,0.0980,0.1002,0.0839,0.0919,0.0993,0.1210,0.1109,0.0878
4,mad,0.1388,0.1642,0.1507,0.1880,0.1307,0.1663,0.1349,0.1384
5,min,0.1248,0.1402,0.1231,0.1582,0.1153,0.1535,0.1361,0.1090
6,sun,0.1166,0.1229,0.0925,0.1255,0.0992,0.1272,0.1581,0.1160


## B3. Length Distribution

Word count distribution synthetic vs real (seed + valid + test) per bahasa.

Apakah synthetic match distribusi panjang real data?

In [12]:
def len_stats(df):
    df = df.copy()
    df['wc'] = df['text'].str.split().str.len()
    return {
        'n': len(df),
        'mean': round(df['wc'].mean(), 1),
        'std': round(df['wc'].std(), 1),
        'min': int(df['wc'].min()),
        'max': int(df['wc'].max()),
        'median': int(df['wc'].median()),
        'p95': int(df['wc'].quantile(0.95)),
    }

rows = []
for lang in LANGS:
    if lang not in syn_dfs:
        continue
    rows.append({'lang': lang, 'split': 'seed', **len_stats(seed_dfs[lang])})
    rows.append({'lang': lang, 'split': 'syn',  **len_stats(syn_dfs[lang])})
    if lang in valid_dfs:
        rows.append({'lang': lang, 'split': 'valid', **len_stats(valid_dfs[lang])})
    if lang in test_dfs:
        rows.append({'lang': lang, 'split': 'test',  **len_stats(test_dfs[lang])})

display(pd.DataFrame(rows))

,lang,split,n,mean,std,min,max,median,p95
0,ace,seed,500,23.9,15.4,4,94,20,56
1,ace,syn,2444,25.2,14.7,1,108,22,55
2,ace,valid,100,23.4,13.9,4,68,20,51
3,ace,test,400,24.1,15.2,4,75,20,57
4,ban,seed,500,23.4,14.6,3,71,19,53
5,ban,syn,2465,24.2,13.2,3,75,21,52
6,ban,valid,100,23.4,13.5,4,64,21,48
7,ban,test,400,23.5,14.5,5,68,20,57
8,bjn,seed,500,23.2,14.6,4,77,19,53
9,bjn,syn,2473,24.0,13.5,5,79,21,52


### B3a. Per-Label PSLM Filter Validation

Validasi apakah PSLM filter (`|gen_len − seed_len| ≤ 1.0σ` per label) bekerja konsisten lintas label. Compute dua metrik per (lang × label):

- **Rasio Standar Deviasi Panjang** ($R_\sigma$) — `std_ratio = syn_std / seed_std`
- **Selisih Rata-rata Panjang** ($\Delta\mu$) — `mean_diff = syn_mean − seed_mean`

**Tujuan**: memastikan match aggregate di B3 bukan akibat averaging antar label dengan distribusi berbeda (neutral umumnya lebih pendek, positive/negative lebih panjang & variatif).

**Verdict criteria** (internal script convention):
- `std_ratio 0.80–1.20` dan `|mean_diff| ≤ 5` → **OK** (filter constrain length drift sesuai design)
- `std_ratio < 0.70` → kalimat sintetis terlalu seragam dalam panjang (over-filtering atau LLM mode-collapse)
- `std_ratio > 1.20` → filter tidak cukup ketat (threshold 1σ terlalu longgar)
- `|mean_diff| > 5` → Generator cenderung menghasilkan kalimat lebih panjang atau lebih pendek dari seed secara konsisten

> **Catatan**: column code di tabel pakai snake_case (`std_ratio`, `mean_diff`) karena variable name di code; di proposal Bab III/IV dilaporkan sebagai *Rasio Standar Deviasi Panjang* ($R_\sigma$) dan *Selisih Rata-rata Panjang* ($\Delta\mu$).

In [13]:
rows = []
for lang in LANGS:
    if lang not in syn_dfs:
        continue
    seed_df, syn_df = seed_dfs[lang].copy(), syn_dfs[lang].copy()
    seed_df['wc'] = seed_df['text'].str.split().str.len()
    syn_df['wc']  = syn_df['text'].str.split().str.len()
    for label in ['negative', 'neutral', 'positive']:
        s_seed = seed_df[seed_df['label'] == label]['wc']
        s_syn  = syn_df[syn_df['label'] == label]['wc']
        if len(s_seed) > 1 and len(s_syn) > 1:
            ratio = round(s_syn.std() / s_seed.std(), 2)
            mean_diff = round(s_syn.mean() - s_seed.mean(), 1)
            verdict = 'OK' if 0.80 <= ratio <= 1.20 and abs(mean_diff) <= 5 else 'CHECK'
            rows.append({
                'lang': lang,
                'label': label,
                'seed_n': len(s_seed),
                'syn_n': len(s_syn),
                'seed_mean': round(s_seed.mean(), 1),
                'syn_mean': round(s_syn.mean(), 1),
                'mean_diff': mean_diff,
                'seed_std': round(s_seed.std(), 1),
                'syn_std': round(s_syn.std(), 1),
                'std_ratio': ratio,
                'verdict': verdict,
            })

display(pd.DataFrame(rows))

,lang,label,seed_n,syn_n,seed_mean,syn_mean,mean_diff,seed_std,syn_std,std_ratio,verdict
0,ace,negative,192,925,22.5,23.8,1.4,15.0,14.3,0.95,OK
1,ace,neutral,119,583,14.5,16.1,1.6,9.6,9.7,1.02,OK
2,ace,positive,189,936,31.2,32.2,1.0,15.4,14.3,0.93,OK
3,ban,negative,192,940,21.9,23.0,1.1,13.8,12.6,0.91,OK
4,ban,neutral,119,583,13.8,15.5,1.7,8.5,8.4,0.99,OK
5,ban,positive,189,942,30.8,30.7,-0.1,14.4,12.8,0.89,OK
6,bjn,negative,192,946,21.9,23.0,1.1,14.1,13.1,0.93,OK
7,bjn,neutral,119,590,13.5,14.7,1.2,8.3,8.2,0.98,OK
8,bjn,positive,189,937,30.6,30.8,0.2,14.4,13.0,0.90,OK
9,jav,negative,192,959,21.8,23.2,1.4,14.2,13.4,0.94,OK


## B4. Label Distribution

Label distribution comparison: synthetic vs seed (NusaX train) per bahasa.

Konfirmasi distribusi synthetic ≈ distribusi seed (no class imbalance shift).

In [14]:
rows = []
for lang in LANGS:
    if lang not in syn_dfs:
        continue
    seed_dist = seed_dfs[lang]['label'].value_counts(normalize=True) * 100
    syn_dist = syn_dfs[lang]['label'].value_counts(normalize=True) * 100
    rows.append({
        'lang': lang,
        'seed_neg%': round(seed_dist.get('negative', 0), 1),
        'syn_neg%':  round(syn_dist.get('negative', 0), 1),
        'diff_neg':  round(syn_dist.get('negative', 0) - seed_dist.get('negative', 0), 1),
        'seed_neu%': round(seed_dist.get('neutral', 0), 1),
        'syn_neu%':  round(syn_dist.get('neutral', 0), 1),
        'diff_neu':  round(syn_dist.get('neutral', 0) - seed_dist.get('neutral', 0), 1),
        'seed_pos%': round(seed_dist.get('positive', 0), 1),
        'syn_pos%':  round(syn_dist.get('positive', 0), 1),
        'diff_pos':  round(syn_dist.get('positive', 0) - seed_dist.get('positive', 0), 1),
    })

display(pd.DataFrame(rows))

,lang,seed_neg%,syn_neg%,diff_neg,seed_neu%,syn_neu%,diff_neu,seed_pos%,syn_pos%,diff_pos
0,ace,38.4,37.8,-0.6,23.8,23.9,0.1,37.8,38.3,0.5
1,ban,38.4,38.1,-0.3,23.8,23.7,-0.1,37.8,38.2,0.4
2,bjn,38.4,38.3,-0.1,23.8,23.9,0.1,37.8,37.9,0.1
3,jav,38.4,38.4,-0.0,23.8,23.8,-0.0,37.8,37.8,0.0
4,mad,38.4,38.4,-0.0,23.8,24.0,0.2,37.8,37.7,-0.1
5,min,38.4,38.4,0.0,23.8,23.8,-0.0,37.8,37.8,-0.0
6,sun,38.4,38.5,0.1,23.8,23.7,-0.1,37.8,37.8,0.0


# Ringkasan untuk Bab IV

Tabel-tabel di atas siap di-paste ke thesis Bab IV.

**Pipeline Metrics (Tujuan poin 1 & 2)**:
- A1: Filter Rate + SV/LV/PSLM Pass Rate per bahasa
- A2 & A3: Reject reason distribution (kualitatif untuk pembahasan)
- A4: Output volume & label distribution match

**Quality Metrics (Tujuan poin 2)**:
- B1: GlotLID purity vs NusaX baseline
- B2: Self-BLEU per bahasa per label
- B3 & B4: Distribusi panjang & label (deskriptif untuk validasi distribusi data)

**Downstream Performance (Tujuan poin 3)** → `finetune_nusabert.ipynb`:
- F1-Score Macro baseline vs +1x/+2x/+3x augmented per bahasa per model
- Accuracy per skenario